# Causal Importance Reanalysis — Phase 2 Group Patching (Llama-3.1-70B-Instruct-4bit (NF4))

Single-model variant (deferred from standard notebook for special architecture/precision pattern).

**Model**: `cmarkea/Meta-Llama-3.1-70B-Instruct-4bit` (NF4 4-bit quantization (bitsandbytes); pre-quantized weights)
**Output dir**: `models/llama-3.1-70b-instruct-4bit/causal_importance_reanalysis/`
**Attn path**: `model.model.layers[L].self_attn.o_proj (Llama-style)`
**VRAM**: ≥ 40 GB (Prospect-measured peak ≈ 37 GB; A100 40GB or G4 95.6GB)

## What this notebook does
Same as the standard 5-model Phase 2 notebook, but for a single model with its specific architecture/precision pattern. Reuses existing artefacts from `models/llama-3.1-70b-instruct-4bit/{10_collection,20_scoring,30_patching}/` to avoid recomputing Phase 0/B/C.1.

1. Compute `causal_imp` per head from existing `individual_head_effects.csv`.
2. Re-classify cells via median split on (causal_imp, perturbation_L2).
3. Build NEW orderings `C_p_imp_asc`, `D_p_imp_asc` (ascending causal_imp).
4. Group-patch at r=0.05, r=0.10, saturation = min(|C_p|, |D_p|).
5. Save `dose_response_v2_causal.csv` matching paper schema + `ratio_label` column.

**Note**: Per-trial Δ target-logit step ≈ 0.125 under NF4. Per-head magnitudes are floor-bound but **group patching at r=0.10 / saturation accumulates well above the floor** (consistent with `v75_70b_exclusion_scope.md`).


## v2 (session-consistent) — 2026-04-26 update

This notebook now extracts `activation_vectors` and `clean_logits` **fresh in the current Colab session** (instead of loading paper Main pipeline's stored `.npz`). This avoids cross-session numerical drift caused by `transformers` / `torch` / HF-revision updates between paper Main run and Phase 2 reanalysis. Output dir is `causal_importance_reanalysis_session_consistent/` to preserve the older `causal_importance_reanalysis/` results for diff comparison.


In [ ]:
# ── Cell 1: GDrive mount → install → HF login → imports ──
from google.colab import drive, runtime
drive.mount('/content/drive')

!pip install -q -U "transformers" "accelerate" "bitsandbytes>=0.46.1" "nnsight" "scipy" "pandas"

import os
os.environ['HF_TOKEN'] = '<YOUR_HF_TOKEN>'
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
from huggingface_hub import login as _hf_login
_hf_login(token=os.environ['HF_TOKEN'])
print('HF_TOKEN set + huggingface_hub.login() called.')

import json, gc, time, math, random, traceback
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import torch

def log(msg):
    print(f'[{datetime.now().strftime("%H:%M:%S")}] {msg}', flush=True)

log(f'Torch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'GPU runtime required'
_props = torch.cuda.get_device_properties(0)
_vram_gb = _props.total_memory / 1024**3
log(f'GPU: {_props.name}  VRAM={_vram_gb:.1f} GB')
assert _vram_gb >= 39.0, (
    f'≥40 GB VRAM required for Llama-70B-NF4 (Prospect-measured peak ≈ 37 GB). '
    f'Current GPU: {_vram_gb:.1f} GB.'
)


In [ ]:
# ── Cell 2: Config ──
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_SPECS = [
    ('cmarkea/Meta-Llama-3.1-70B-Instruct-4bit', 'llama-3.1-70b-instruct-4bit'),
]
MODEL_DTYPE = 'nf4'

PIPELINE_VERSION    = 'v7.5_causal_imp_phase2'
ANALYSIS_NAME       = 'causal_importance_reanalysis'
ANALYSIS_NAME_V2    = 'causal_importance_reanalysis_v2_session_consistent'  # output dir for session-consistent v2 method
PATCH_MECHANISM     = 'input_overwrite_v5_1_pattern'  # Pattern A
RSA_METRIC          = 'cosine_similarity_rdm'         # paper-side rsa_max metric (referenced for context)

# Group patching trial-batch size (paper main convention; OOM ladder fallback)
PATCH_BATCH_SIZE    = 16
PATCH_BATCH_LADDER  = [16, 12, 8, 4, 2, 1]

# Trial structure (matches paper Appendix A)
CREL_LIST  = ['SAME', 'OPP', 'MORE', 'LESS']
CREL_PAIRS = [
    ('SAME', 'OPP'), ('SAME', 'MORE'), ('SAME', 'LESS'),
    ('OPP',  'MORE'), ('OPP',  'LESS'), ('MORE',  'LESS'),
]
ATTR_DIMS  = ['P', 'Q']

# Orderings to evaluate (causal_imp-based new cells)
# To extend: add 'hihp_p_imp_desc', 'hilp_p_imp_desc' for paper-Table-1-style parallel.
ORDERINGS  = ['C_p_imp_asc', 'D_p_imp_asc']

# Ratios: 0.05, 0.10, saturation. saturation = min(|C_p|, |D_p|) / total_heads (per-model dynamic).
RATIO_LABELS = ['r0.05', 'r0.10', 'saturation']

DRIVE_BASE   = '/content/drive/MyDrive/WCC'
MODELS_BASE  = f'{DRIVE_BASE}/models'

log(f'Pipeline: {PIPELINE_VERSION}  Analysis: {ANALYSIS_NAME}')
log(f'Models: {len(MODEL_SPECS)} (standard bf16)')
log(f'Orderings: {ORDERINGS}')
log(f'Ratios: {RATIO_LABELS}')
log(f'PATCH_BATCH_SIZE={PATCH_BATCH_SIZE} (ladder={PATCH_BATCH_LADDER})')


In [ ]:
# ── Cell 3: Utility helpers ──
import psutil as _psutil_mem

def _mem_status():
    vm = _psutil_mem.virtual_memory()
    host_gb = vm.used / 1024**3
    host_total = vm.total / 1024**3
    gpu_alloc = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
    gpu_peak  = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
    return f'host={host_gb:.1f}/{host_total:.1f}GB gpu={gpu_alloc:.1f}GB peak={gpu_peak:.1f}GB'

def _sweep_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def ensure_dir(p):
    os.makedirs(p, exist_ok=True); return p

def pair_sort_key(s):
    # 'pair0'..'pair9' → (0..9). Used to keep pair ordering deterministic.
    if isinstance(s, str) and s.startswith('pair'):
        try: return int(s[4:])
        except: return s
    return s

def load_npz_dict(path, label):
    npz = np.load(path)
    out = {k: npz[k] for k in npz.files}
    npz.close()
    return out

def unload_model(model):
    try: del model
    except Exception: pass
    _sweep_memory()

def ratio_to_k_floor(ratio, total_heads):
    """Paper Main convention: k = floor(r × total_heads). Not ceil. (At least 1.)"""
    return max(1, int(float(ratio) * float(total_heads)))

def build_patch_trials(trials_df, pair_ids):
    """Build the 120 (crel_pair × pair_id × attr_dim) Rule-α (src→tgt) patch trials."""
    trial_by_key = {(row['crel'], row['pair_id'], row['attr_dim']): row
                    for row in trials_df.to_dict('records')}
    out = []
    for src_crel, tgt_crel in CREL_PAIRS:
        for pair_id in sorted(pair_ids, key=pair_sort_key):
            for attr_dim in ATTR_DIMS:
                k_src = (src_crel, pair_id, attr_dim)
                k_tgt = (tgt_crel, pair_id, attr_dim)
                if k_src not in trial_by_key or k_tgt not in trial_by_key:
                    continue
                src_row = trial_by_key[k_src]
                tgt_row = trial_by_key[k_tgt]
                out.append({
                    'src_tid'  : src_row['trial_id'],
                    'tgt_tid'  : tgt_row['trial_id'],
                    'crel_pair': f'{src_crel}-{tgt_crel}',
                    'pair_id'  : pair_id,
                    'attr_dim' : attr_dim,
                    'prompt'   : tgt_row['prompt'],
                    'tgt_tok'  : int(tgt_row['correct_first_subtoken_id']) if 'correct_first_subtoken_id' in tgt_row else None,
                    'src_tok'  : int(src_row['correct_first_subtoken_id']) if 'correct_first_subtoken_id' in src_row else None,
                })
    return out


In [ ]:
# ── Cell 4: Model load + group patching helpers (Llama-70B-Instruct NF4 4-bit) ──
import nnsight as nns
from nnsight import LanguageModel
log(f'nnsight version: {nns.__version__}')


def load_model_standard(model_id):
    """70B-NF4: pre-quantized model. Do NOT pass dtype (uses embedded quant_config). Llama path."""
    log(f'Loading {model_id}  (NF4 4-bit, Llama path; uses embedded quant_config)')
    t0 = time.time()
    model = LanguageModel(model_id, device_map='auto')  # NF4 model: no dtype kwarg
    cfg = model.config
    arch = {
        'num_layers' : cfg.num_hidden_layers,
        'num_heads'  : cfg.num_attention_heads,
        'head_dim'   : getattr(cfg, 'head_dim', cfg.hidden_size // cfg.num_attention_heads),
        'hidden_size': cfg.hidden_size,
        'vocab_size' : cfg.vocab_size,
    }
    arch['total_heads'] = arch['num_layers'] * arch['num_heads']
    log(f'  arch: L={arch["num_layers"]} H={arch["num_heads"]} D={arch["head_dim"]} '
        f'total={arch["total_heads"]}  vocab={arch["vocab_size"]}  load={time.time()-t0:.1f}s  '
        f'{_mem_status()}')
    return model, arch


@torch.no_grad()
def clean_logits_batched(model, prompts):
    if not prompts:
        return np.empty((0,), dtype=np.float32)
    with model.trace(prompts) as tracer:
        logits = model.output.logits[:, -1, :].save()
    return logits.detach().float().cpu().numpy()


@torch.no_grad()
def patched_logits_batched_trials(model, arch, prompts, source_vecs, patch_heads):
    """70B-NF4 path: model.model.layers[L].self_attn.o_proj (Llama-style). bf16 src tensor."""
    B = len(prompts)
    if B == 0:
        return np.empty((0, arch['vocab_size']), dtype=np.float32)
    H, D = arch['num_heads'], arch['head_dim']
    src_stack = torch.as_tensor(np.stack(source_vecs), dtype=torch.bfloat16, device='cuda')
    sorted_patches = sorted(patch_heads, key=lambda lh: (int(lh[0]), int(lh[1])))
    with model.trace(prompts) as tracer:
        for layer_idx, head_idx in sorted_patches:
            proj = model.model.layers[int(layer_idx)].self_attn.o_proj
            col_s = int(head_idx) * D
            proj.input[:, -1, col_s:col_s + D] = src_stack[:, int(layer_idx), int(head_idx)]
        logits = model.output.logits[:, -1, :].save()
    return logits.detach().float().cpu().numpy()


def _call_with_batch_ladder(fn, *args, **kwargs):
    global PATCH_BATCH_SIZE
    ladder = list(PATCH_BATCH_LADDER)
    if PATCH_BATCH_SIZE not in ladder:
        ladder = [PATCH_BATCH_SIZE] + [b for b in ladder if b < PATCH_BATCH_SIZE]
    ladder_ptr = ladder.index(PATCH_BATCH_SIZE) if PATCH_BATCH_SIZE in ladder else 0
    while True:
        b = ladder[ladder_ptr]
        try:
            fn(b, *args, **kwargs)
            PATCH_BATCH_SIZE = b
            return b
        except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
            msg = str(exc).lower()
            if 'out of memory' not in msg and not isinstance(exc, torch.cuda.OutOfMemoryError):
                raise
            _sweep_memory()
            if ladder_ptr >= len(ladder) - 1:
                raise
            ladder_ptr += 1
            log(f'  OOM at batch={b} → ladder down to {ladder[ladder_ptr]}  ({_mem_status()})')



@torch.no_grad()
def collect_session_npz(model, arch, trials_df):
    """v5.1.1 proven pattern (nnsight 0.6.3): plain Python assignments inside `with model.trace()`
    do NOT escape the worker frame; only .save() return values do. So we stack-then-save once.
    """
    L, H, D = arch['num_layers'], arch['num_heads'], arch['head_dim']
    layers_obj = model.model.layers
    acts_dict, clean_logits_dict = {}, {}
    t0 = time.time()
    trials = trials_df.to_dict('records')
    for t in trials:
        prompt = t['prompt']
        with model.trace(prompt) as tracer:
            head_inputs = []
            for li in range(L):
                attn = layers_obj[li].self_attn.o_proj.input[0, -1, :]
                head_inputs.append(attn.view(H, D).cpu())
            stacked_proxy = torch.stack(head_inputs).save()       # (L, H, D)
            logits_proxy  = model.output.logits[0, -1, :].save()  # (vocab,)
        stacked = getattr(stacked_proxy, 'value', stacked_proxy)
        logits  = getattr(logits_proxy,  'value', logits_proxy)
        acts_dict[t['trial_id']]         = stacked.detach().float().cpu().numpy().astype(np.float32, copy=False)
        clean_logits_dict[t['trial_id']] = logits.detach().float().cpu().numpy().astype(np.float32, copy=False)
    log(f'  Fresh npz extraction: {len(acts_dict)} trials in {len(trials)} traces, {time.time()-t0:.1f}s')
    return acts_dict, clean_logits_dict


In [ ]:
# ── Cell 5: Phase 2 implementation (causal_imp re-classification + group patching) ──

def compute_per_head_causal_imp(ihe_df):
    """Per-head causal importance = mean(|Δ target logit|) over 120 trials."""
    ihe_df = ihe_df.copy()
    ihe_df['abs_dT'] = ihe_df['delta_target_logit_signed'].abs()
    out = ihe_df.groupby(['layer','head'])['abs_dT'].mean().reset_index()
    out.columns = ['layer','head','causal_imp']
    return out


def reclassify_cells_causal(causal_df, cc_df):
    """Median split on (causal_imp, perturbation_L2). Add cell_new_abs column."""
    merged = causal_df.merge(cc_df[['layer','head','perturbation_L2']], on=['layer','head'], how='inner')
    valid = merged['causal_imp'].notna() & merged['perturbation_L2'].notna()
    if not valid.any():
        raise RuntimeError('No valid heads for cell classification')
    med_imp  = float(merged.loc[valid, 'causal_imp'].median())
    med_pert = float(merged.loc[valid, 'perturbation_L2'].median())
    cell = pd.Series([None]*len(merged), index=merged.index, dtype=object)
    cell[valid & (merged['causal_imp'] >= med_imp) & (merged['perturbation_L2'] >= med_pert)] = 'hihp_p'
    cell[valid & (merged['causal_imp'] >= med_imp) & (merged['perturbation_L2'] <  med_pert)] = 'hilp_p'
    cell[valid & (merged['causal_imp'] <  med_imp) & (merged['perturbation_L2'] >= med_pert)] = 'C_p'
    cell[valid & (merged['causal_imp'] <  med_imp) & (merged['perturbation_L2'] <  med_pert)] = 'D_p'
    merged['cell_new_abs'] = cell
    merged['causal_imp_median']     = med_imp
    merged['perturbation_median']   = med_pert
    return merged, {'causal_imp_median': med_imp, 'perturbation_median': med_pert}


def build_orderings_causal(cell_df):
    """Sort within C_p, D_p by causal_imp ASC (paper convention). Tie-break by (layer, head)."""
    out = {}
    for cell_letter in ('C_p', 'D_p'):
        sub = cell_df[cell_df['cell_new_abs']==cell_letter].dropna(subset=['causal_imp']).copy()
        srt = sub.sort_values(['causal_imp','layer','head'],
                              ascending=[True, True, True], kind='mergesort')
        out[f'{cell_letter}_imp_asc'] = list(zip(srt['layer'].astype(int), srt['head'].astype(int)))
    return out


def run_group_patching_for_combo(model, arch, patch_trials, all_vecs, clean_logits_dict, active_heads):
    """Run group patching for one (group, ratio) combo across all 120 trials. Returns list of row dicts."""
    if not active_heads:
        # Empty patch → no-op; return clean values
        return [{
            'crel_pair': t['crel_pair'],
            'pair_id': t['pair_id'],
            'attr_dim': t['attr_dim'],
            'delta_target_logit_signed': 0.0,
            'delta_source_logit_signed': 0.0,
            'argmax_token_id_clean': int(np.argmax(clean_logits_dict[t['tgt_tid']])),
            'argmax_token_id_patched': int(np.argmax(clean_logits_dict[t['tgt_tid']])),
        } for t in patch_trials]

    rows = []
    batch_start = 0
    while batch_start < len(patch_trials):
        def _do_batch(B, bs=batch_start):
            batch = patch_trials[bs:bs + B]
            if not batch: return
            prompts     = [t['prompt'] for t in batch]
            source_vecs = [all_vecs[t['src_tid']] for t in batch]
            patched_logits = patched_logits_batched_trials(model, arch, prompts, source_vecs, active_heads)
            for idx, t in enumerate(batch):
                clean_lg   = clean_logits_dict[t['tgt_tid']]
                patched_lg = patched_logits[idx]
                tgt_tok    = int(t['tgt_tok'])
                src_tok    = int(t['src_tok'])
                rows.append({
                    'crel_pair': t['crel_pair'],
                    'pair_id'  : t['pair_id'],
                    'attr_dim' : t['attr_dim'],
                    'delta_target_logit_signed': float(patched_lg[tgt_tok] - clean_lg[tgt_tok]),
                    'delta_source_logit_signed': float(patched_lg[src_tok] - clean_lg[src_tok]),
                    'argmax_token_id_clean'    : int(np.argmax(clean_lg)),
                    'argmax_token_id_patched'  : int(np.argmax(patched_lg)),
                })
        used_b = _call_with_batch_ladder(_do_batch)
        batch_start += used_b
    return rows


def run_phase2_for_model(model_id, model_short):
    """Phase 2 for one model: re-classify, build orderings, run group patching, save outputs."""
    base    = f'{MODELS_BASE}/{model_short}'
    out_dir = ensure_dir(f'{base}/{ANALYSIS_NAME_V2}')

    out_csv         = f'{out_dir}/dose_response_v2_causal.csv'
    out_cfg         = f'{out_dir}/config.json'
    out_per_head    = f'{out_dir}/per_head_causal_imp.csv'
    out_cell_class  = f'{out_dir}/cell_classification_causal.csv'

    # Required input artefacts
    req_paths = {
        'individual_head_effects': f'{base}/30_patching/individual_head_effects.csv',
        'cell_classification':     f'{base}/20_scoring/cell_classification.csv',
        'trial_definitions':       f'{base}/10_collection/trial_definitions.csv',
        'activation_vectors':      f'{base}/10_collection/activation_vectors.npz',
        'clean_logits':            f'{base}/10_collection/clean_logits.npz',
    }
    for name, p in req_paths.items():
        if not os.path.exists(p):
            raise FileNotFoundError(f'{model_short}: missing {name} at {p}')

    log(f'  Loading inputs (session-consistent: stored npz NOT used; will extract fresh)...')
    ihe          = pd.read_csv(req_paths['individual_head_effects'])
    cc           = pd.read_csv(req_paths['cell_classification'])
    trials_df    = pd.read_csv(req_paths['trial_definitions'])
    log(f'    ihe={len(ihe)} rows  cc={len(cc)} heads  trials={len(trials_df)}')
    # NOTE: activation_vectors.npz / clean_logits.npz from paper Main pipeline session are NOT loaded.
    # We re-extract them fresh below (after model load) for session-internal consistency.

    # Recompute causal_imp + re-classify cells
    causal_df = compute_per_head_causal_imp(ihe)
    causal_df.to_csv(out_per_head, index=False)
    cell_df, thresholds = reclassify_cells_causal(causal_df, cc)
    cell_df_for_save = cell_df[['layer','head','causal_imp','perturbation_L2','cell_new_abs',
                                'causal_imp_median','perturbation_median']].copy()
    cell_df_for_save.to_csv(out_cell_class, index=False)

    cell_counts = {c: int((cell_df['cell_new_abs']==c).sum())
                   for c in ['hihp_p','hilp_p','C_p','D_p']}
    log(f'    causal_imp median={thresholds["causal_imp_median"]:.5f}  '
        f'pert median={thresholds["perturbation_median"]:.5f}')
    log(f'    cell counts: {cell_counts}')

    # Build orderings + patch_trials
    orderings_map = build_orderings_causal(cell_df)
    log(f'    orderings: {[f"{k}=({len(v)})" for k,v in orderings_map.items()]}')
    pair_ids = sorted(trials_df['pair_id'].unique().tolist(), key=pair_sort_key)
    patch_trials = build_patch_trials(trials_df, pair_ids)
    log(f'    patch_trials: {len(patch_trials)} (expected {len(CREL_PAIRS)*len(pair_ids)*len(ATTR_DIMS)})')

    # Verify trials_df has correct_first_subtoken_id (or compute it)
    if 'correct_first_subtoken_id' not in trials_df.columns:
        log('    NOTE: trial_definitions lacks correct_first_subtoken_id; computing from tokenizer.')
        # Fall back to tokenizer encoding (model required)

    # Saturation k = min(|C_p|, |D_p|)
    n_C_p = len(orderings_map['C_p_imp_asc'])
    n_D_p = len(orderings_map['D_p_imp_asc'])
    sat_k = min(n_C_p, n_D_p)

    # Load model (needed for fresh extraction below)
    model, arch = load_model_standard(model_id)
    total_heads = arch['total_heads']
    sat_ratio = sat_k / total_heads if total_heads > 0 else 0.0
    log(f'    saturation: k={sat_k}  ratio={sat_ratio:.4f}  (|C_p|={n_C_p}, |D_p|={n_D_p})')

    # Fresh extraction: activation_vectors + clean_logits computed in CURRENT session
    # → no cross-session drift. (Old sanity check vs stored npz is now obsolete and removed.)
    model_for_extract, arch_for_extract = model, arch
    all_vecs, clean_logits_dict = collect_session_npz(model_for_extract, arch_for_extract, trials_df)
    try:
        np.savez_compressed(f'{out_dir}/activation_vectors_session.npz', **all_vecs)
        np.savez_compressed(f'{out_dir}/clean_logits_session.npz', **clean_logits_dict)
        log(f'  Saved fresh npz to {out_dir}/(activation_vectors|clean_logits)_session.npz')
    except Exception as _save_e:
        log(f'  WARN fresh npz save failed: {type(_save_e).__name__}: {_save_e}')
    log(f'    fresh acts keys={len(all_vecs)}  fresh clean keys={len(clean_logits_dict)}')
    # Post-extraction sanity: every trial_id we will patch must be in fresh dicts.
    # By construction collect_session_npz iterates ALL trials_df rows, so this should
    # Safety net so any key-mismatch surfaces loudly rather than silently.
    missing_src = [t['src_tid'] for t in patch_trials if t['src_tid'] not in all_vecs]
    missing_tgt = [t['tgt_tid'] for t in patch_trials if t['tgt_tid'] not in clean_logits_dict]
    if missing_src or missing_tgt:
        raise RuntimeError(f'Missing src tids in act_vecs ({len(missing_src)}) or tgt tids in clean ({len(missing_tgt)})')


    # trial_definitions.csv lacks correct_first_subtoken_id column — compute via tokenizer.
    # Use prefix-subtraction (FR notebook convention) for deterministic first-subtoken extraction
    # across BPE / SentencePiece / GPT-2 byte-BPE tokenizers.
    if patch_trials[0]['tgt_tok'] is None:
        log('    Computing correct_first_subtoken_id via tokenizer (prefix-subtraction convention)...')
        tokenizer = model.tokenizer
        ANCHOR = 'A:'   # paper Appendix A.3 prompt format ends with 'A:' so this is the natural anchor
        anchor_ids = tokenizer.encode(ANCHOR, add_special_tokens=False)
        trials_indexed = trials_df.set_index('trial_id')
        for t in patch_trials:
            tgt_text  = str(trials_indexed.loc[t['tgt_tid'], 'correct_answer'])
            src_text  = str(trials_indexed.loc[t['src_tid'], 'correct_answer'])
            tgt_full  = tokenizer.encode(ANCHOR + ' ' + tgt_text, add_special_tokens=False)
            src_full  = tokenizer.encode(ANCHOR + ' ' + src_text, add_special_tokens=False)
            t['tgt_tok'] = int(tgt_full[len(anchor_ids)])
            t['src_tok'] = int(src_full[len(anchor_ids)])
        sample_tids = list({t['tgt_tok'] for t in patch_trials[:6]})
        log(f'    sample tgt_tok ids: {sample_tids}')

    # Resume: load existing CSV
    if os.path.exists(out_csv):
        existing_df = pd.read_csv(out_csv)
        if 'group' in existing_df.columns and 'ratio_label' in existing_df.columns:
            completed = set(zip(existing_df['group'].astype(str), existing_df['ratio_label'].astype(str)))
        else:
            completed = set()
        rows_buffer = list(existing_df.to_dict('records'))
        log(f'    Resume: {len(completed)} (group, ratio_label) combos already in CSV  rows={len(rows_buffer)}')
    else:
        existing_df = pd.DataFrame()
        completed = set()
        rows_buffer = []

    # Run combos: 2 orderings × 3 ratio_labels = 6 combos per model
    csv_columns = ['model','group','ratio','ratio_label','k','k_actual',
                   'crel_pair','pair_id','attr_dim',
                   'delta_target_logit_signed','delta_source_logit_signed',
                   'argmax_token_id_clean','argmax_token_id_patched',
                   'n_cell_C_p','n_cell_D_p','n_cell_hihp_p','n_cell_hilp_p',
                   'causal_imp_median','perturbation_median']

    t_phase = time.time()
    for ratio_label in RATIO_LABELS:
        if ratio_label == 'r0.05':
            base_ratio = 0.05
            k_base = ratio_to_k_floor(0.05, total_heads)
        elif ratio_label == 'r0.10':
            base_ratio = 0.10
            k_base = ratio_to_k_floor(0.10, total_heads)
        elif ratio_label == 'saturation':
            base_ratio = sat_ratio
            k_base = sat_k
        else:
            raise ValueError(f'Unknown ratio_label: {ratio_label}')

        for ordering_name in ORDERINGS:
            if (ordering_name, ratio_label) in completed:
                log(f'    SKIP {ordering_name} {ratio_label}  (already in CSV)')
                continue

            ordered_heads = orderings_map[ordering_name]
            k_actual = min(k_base, len(ordered_heads))
            active_heads = ordered_heads[:k_actual]

            t_combo = time.time()
            combo_rows = run_group_patching_for_combo(model, arch, patch_trials,
                                                     all_vecs, clean_logits_dict, active_heads)
            dt = time.time() - t_combo

            # Augment with metadata + append to buffer
            for r in combo_rows:
                r.update({
                    'model': model_short,
                    'group': ordering_name,
                    'ratio': float(base_ratio),
                    'ratio_label': ratio_label,
                    'k': int(k_base),
                    'k_actual': int(k_actual),
                    'n_cell_C_p': cell_counts['C_p'],
                    'n_cell_D_p': cell_counts['D_p'],
                    'n_cell_hihp_p': cell_counts['hihp_p'],
                    'n_cell_hilp_p': cell_counts['hilp_p'],
                    'causal_imp_median': thresholds['causal_imp_median'],
                    'perturbation_median': thresholds['perturbation_median'],
                })
            rows_buffer.extend(combo_rows)
            completed.add((ordering_name, ratio_label))

            # Save per-combo (small writes; 6 combos × 5 models = 30 writes; well below Drive throttling)
            df_save = pd.DataFrame(rows_buffer)
            # Order columns; preserve unknown extras
            ordered_cols = [c for c in csv_columns if c in df_save.columns]
            extra_cols = [c for c in df_save.columns if c not in csv_columns]
            df_save = df_save[ordered_cols + extra_cols]
            df_save.to_csv(out_csv, index=False)

            mean_abs = float(np.nanmean([abs(r['delta_target_logit_signed']) for r in combo_rows]))
            log(f'    DONE {ordering_name} {ratio_label}  k={k_actual}  '
                f'mean|Δ_tgt|={mean_abs:.5f}  {dt:.1f}s  {_mem_status()}')
            _sweep_memory()

    # Save config
    cfg = {
        'pipeline_version'   : PIPELINE_VERSION,
        'analysis_name'      : ANALYSIS_NAME,
        'model_id'           : model_id,
        'model_short'        : model_short,
        'model_dtype'        : MODEL_DTYPE,
        'orderings'          : ORDERINGS,
        'ratio_labels'       : RATIO_LABELS,
        'r_values'           : {'r0.05': 0.05, 'r0.10': 0.10, 'saturation': sat_ratio},
        'k_values'           : {
            'r0.05': ratio_to_k_floor(0.05, total_heads),
            'r0.10': ratio_to_k_floor(0.10, total_heads),
            'saturation': sat_k,
        },
        'cell_counts'        : cell_counts,
        'thresholds'         : thresholds,
        'patch_batch_size_used': PATCH_BATCH_SIZE,
        'total_heads'        : total_heads,
        'n_patch_trials'     : len(patch_trials),
        'phase2_elapsed_sec' : round(time.time() - t_phase, 2),
        'completed_at'       : datetime.now().isoformat(),
    }
    with open(out_cfg, 'w') as f:
        json.dump(cfg, f, indent=2)
    log(f'  Saved config: {out_cfg}')

    unload_model(model)
    return cfg


In [ ]:
# ── Cell 6: Main loop over 5 standard models ──
all_success = True
results_summary = []
for MODEL_ID, MODEL_SHORT in MODEL_SPECS:
    log('=' * 70)
    log(f'START: {MODEL_SHORT}  ({MODEL_ID})  initial mem={_mem_status()}')
    t_model = time.time()
    try:
        cfg = run_phase2_for_model(MODEL_ID, MODEL_SHORT)
        results_summary.append({
            'model': MODEL_SHORT,
            'ok': True,
            'elapsed_min': round((time.time() - t_model) / 60, 2),
            'phase2_elapsed_sec': cfg.get('phase2_elapsed_sec'),
            'cell_counts': cfg.get('cell_counts'),
        })
        log(f'  DONE: {MODEL_SHORT}  total={(time.time()-t_model)/60:.1f} min  '
            f'final_mem={_mem_status()}')
    except Exception as e:
        log(f'  FAILED: {MODEL_SHORT}  {type(e).__name__}: {str(e)[:600]}')
        traceback.print_exc()
        all_success = False
        results_summary.append({
            'model': MODEL_SHORT, 'ok': False,
            'error': f'{type(e).__name__}: {str(e)[:300]}',
            'elapsed_min': round((time.time() - t_model) / 60, 2),
        })

log('=' * 70)
log(f'ALL MODELS PROCESSED.  overall_success={all_success}')
log('Summary:')
for r in results_summary:
    if r['ok']:
        log(f'  {r["model"]:<22} OK   elapsed={r["elapsed_min"]} min  '
            f'phase2_sec={r.get("phase2_elapsed_sec")}  cells={r.get("cell_counts")}')
    else:
        log(f'  {r["model"]:<22} FAIL elapsed={r["elapsed_min"]} min  err={r.get("error")}')


In [ ]:
# ── Cell 7: Audio beep + runtime.unassign() only on full success ──
try:
    from IPython.display import Audio, display
    sr = 44100; _t = np.linspace(0, 1, sr)
    display(Audio(0.5 * np.sin(2 * np.pi * 880 * _t), rate=sr, autoplay=True))
except Exception as e:
    log(f'beep failed: {e}')

if all_success:
    log('All 5 standard models complete. Unassigning Colab runtime.')
    runtime.unassign()
else:
    log('Some models failed; keeping runtime alive for inspection.')
